In [2]:
import pandas as pd

In [3]:

ceas = pd.read_csv("../Processed-Data/ceas_cleaned.csv")
enron = pd.read_csv("../Processed-Data/enron_cleaned.csv")
ling = pd.read_csv("../Processed-Data/ling_cleaned.csv")
nigerian = pd.read_csv("../Processed-Data/nigerian_fraud_cleaned.csv")

combined_df = pd.concat(
    [ceas, enron, ling, nigerian],
    ignore_index=True
)

combined_df = combined_df.sample(frac=1, random_state=42).reset_index(drop=True)

print("Combined Shape:", combined_df.shape)
print(combined_df["label"].value_counts())

combined_df.to_csv("../Processed-Data/final_combined_dataset.csv", index=False)

Combined Shape: (74538, 2)
label
1    39374
0    35164
Name: count, dtype: int64


In [4]:
from sklearn.model_selection import train_test_split

X = combined_df["email_text"]
y = combined_df["label"]

X_train,X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.2,
    random_state = 42,
    stratify = y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

print("\nTraining label distribution:")
print(y_train.value_counts())

print("\nTesting label distribution:")
print(y_test.value_counts())

X_train shape: (59630,)
X_test shape: (14908,)
y_train shape: (59630,)
y_test shape: (14908,)

Training label distribution:
label
1    31499
0    28131
Name: count, dtype: int64

Testing label distribution:
label
1    7875
0    7033
Name: count, dtype: int64


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=2000,
    ngram_range=(1, 2)
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("X_train_tfidf shape:", X_train_tfidf.shape)
print("X_test_tfidf shape:", X_test_tfidf.shape)
    

X_train_tfidf shape: (59630, 2000)
X_test_tfidf shape: (14908, 2000)


In [7]:
from sklearn.svm import LinearSVC

svm_model = LinearSVC()
svm_model.fit(X_train_tfidf, y_train)

svm_preds = svm_model.predict(X_test_tfidf)

In [9]:
from sklearn.neighbors import KNeighborsClassifier

# smaller subset for KNN
X_train_small = X_train.sample(3000, random_state=42)
y_train_small = y_train.loc[X_train_small.index]

X_test_small = X_test.sample(500, random_state=42)
y_test_small = y_test.loc[X_test_small.index]

# lighter TF-IDF for KNN
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_knn = TfidfVectorizer(
    stop_words="english",
    max_features=1000,
    ngram_range=(1, 1)
)

X_train_small_tfidf = tfidf_knn.fit_transform(X_train_small)
X_test_small_tfidf = tfidf_knn.transform(X_test_small)

print("KNN train shape:", X_train_small_tfidf.shape)
print("KNN test shape:", X_test_small_tfidf.shape)

knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train_small_tfidf, y_train_small)

knn_preds = knn_model.predict(X_test_small_tfidf)

KNN train shape: (3000, 1000)
KNN test shape: (500, 1000)


In [12]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("===== SVM RESULTS =====")
print("Accuracy:", accuracy_score(y_test, svm_preds))
print("\nClassification Report:\n", classification_report(y_test, svm_preds))
print("Confusion Matrix:\n", confusion_matrix(y_test, svm_preds))

print("===== KNN RESULTS =====")
print("Accuracy:", accuracy_score(y_test_small, knn_preds))
print("\nClassification Report:\n", classification_report(y_test_small, knn_preds))
print("Confusion Matrix:\n", confusion_matrix(y_test_small, knn_preds))

===== SVM RESULTS =====
Accuracy: 0.9777300778105715

Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.97      0.98      7033
           1       0.98      0.98      0.98      7875

    accuracy                           0.98     14908
   macro avg       0.98      0.98      0.98     14908
weighted avg       0.98      0.98      0.98     14908

Confusion Matrix:
 [[6842  191]
 [ 141 7734]]
===== KNN RESULTS =====
Accuracy: 0.674

Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.34      0.51       247
           1       0.61      1.00      0.76       253

    accuracy                           0.67       500
   macro avg       0.80      0.67      0.63       500
weighted avg       0.80      0.67      0.63       500

Confusion Matrix:
 [[ 84 163]
 [  0 253]]
